In [1]:
# !aws s3 sync s3://translinkdata ../data/lambda_sync

In [2]:
import pandas as pd
import numpy as np
import glob
import polars as pl
import os
import hashlib
from datetime import datetime
import altair as alt

# alt.data_transformers.enable("vegafusion")

In [3]:
def generate_file_id(filename):
    """Generate a unique ID for a file using SHA-256 hash."""
    return hashlib.sha256(os.path.basename(filename).encode()).hexdigest()[:16]

def process_translink_realtime_polars(matching_files):
    """Read local parquet files based on path and pattern.

    Args:
        data_path (str): Local path to the directory containing parquet files
        pattern (str): Either 'position' or 'realtime'

    Returns:
        pd.DataFrame: Combined data from all matching parquet files
    """


    schema = {
        "id": pl.Utf8,
        "is_deleted": pl.Boolean,
        "trip_id": pl.Utf8,
        "start_date": pl.Utf8,
        "schedule_relationship": pl.Int64,
        "route_id": pl.Utf8,
        "direction_id": pl.Int64,
        "vehicle_id": pl.Utf8,
        "vehicle_label": pl.Utf8,
        "current_datetime": pl.Datetime("ns"),
        "stop_sequence": pl.Int64,
        "stop_id": pl.Utf8,
        "arrival_delay": pl.Int64,
        "arrival_time": pl.Float64,
        "departure_delay": pl.Float64,
        "departure_time": pl.Float64,
        "stop_schedule_relationship": pl.Int64,
    }
    collected_df = pl.concat(
        [
            pl.scan_parquet(f)
            .with_columns([pl.col(col).cast(dtype) for col, dtype in schema.items()])
            .with_columns(scrape_id=pl.lit(generate_file_id(f)))
            for f in matching_files
        ],
    )

    # collected_df = pl.scan_parquet(matching_files, schema=schema)

    formatted_df = (
        collected_df.with_columns(
            pl.from_epoch(pl.col("arrival_time"), time_unit="s")
            .dt.convert_time_zone("America/Vancouver")
            .alias("arrival_time"),
        )
        .with_columns(
            pl.from_epoch(pl.col("departure_time"), time_unit="s")
            .dt.convert_time_zone("America/Vancouver")
            .alias("departure_time")
        )
        .with_columns(
            pl.col("current_datetime")
            .dt.convert_time_zone("America/Vancouver")
            .alias("current_datetime")
        )
        .with_columns(pl.col("current_datetime").dt.date().alias("current_date"))
    )
    return formatted_df

In [4]:
def read_weather_data_polars():
    weather_files = glob.glob("../data/lambda_sync/weather_data/*.parquet")
    print(weather_files)

    weather_df = pl.scan_parquet(
        weather_files,
    )
    weather_df = (
        (
            weather_df.rename(lambda x: x.replace(".", ""))
            .with_columns(
                pl.col("nowobsTime")
                .cast(pl.Datetime("ns"))
                .dt.convert_time_zone("America/Vancouver")
            )
            .with_columns(
                pl.col("updateTime")
                .cast(pl.Datetime("ns"))
                .dt.convert_time_zone("America/Vancouver")
            )
            .with_columns(pl.col("nowtemp").cast(pl.Int64))
            .with_columns(pl.col("nowprecip").cast(pl.Float64))
        )
        .with_columns(
            nowobsTime_three_hour_bucket=pl.col("nowobsTime").dt.truncate("3h")
        )
        .select(
            [
                # "updateTime",
                # "nowobsTime",
                "nowobsTime_three_hour_bucket",
                "nowtemp",
                "nowtext",
                "nowprecip",
            ]
        )
    )

    return weather_df


# read_weather_data_polars().head().collect()

In [5]:
def read_file_names(data_path, pattern):
    # data_path = "../data/lambda_sync/raw_data"
    # pattern = "realtime"
    # Create full pattern and print it for debugging
    glob_pattern = os.path.join(data_path, f"*_{pattern}.parquet")
    print(f"Searching with pattern: {glob_pattern}")

    # List all files in directory for debugging
    # print(f"Files in directory: {os.listdir(data_path)}")

    # Find all parquet files matching the pattern
    matching_files = glob.glob(glob_pattern)
    # print(f"Found files: {matching_files}")

    if not matching_files:
        raise ValueError(f"No parquet files found matching pattern '{pattern}'")

    # Read and combine all matching parquet files
    
    batch_size = 400

    print(len(matching_files)/batch_size)

    buckets = []
    bucket_stats = {}

    for i in range(0, len(matching_files), batch_size):
        buckets.append(matching_files[i:i+batch_size])
        # bucket_stats[i] = len(matching_files[i:i+batch_size])


    # print(bucket_stats)
    return buckets



In [6]:
def process_translink_position_polars(matching_files):
    """Read local parquet files based on path and pattern.

    Args:
        data_path (str): Local path to the directory containing parquet files
        pattern (str): Either 'position' or 'realtime'

    Returns:
        pd.DataFrame: Combined data from all matching parquet files
    """
    # data_path = "../data/lambda_sync/raw_data"
    # pattern = "position"
    # # Create full pattern and print it for debugging
    # glob_pattern = os.path.join(data_path, f"*_{pattern}.parquet")
    # print(f"Searching with pattern: {glob_pattern}")

    # # List all files in directory for debugging
    # print(f"Files in directory: {os.listdir(data_path)}")

    # # Find all parquet files matching the pattern
    # matching_files = glob.glob(glob_pattern)
    # print(f"Found files: {matching_files}")

    # if not matching_files:
    #     raise ValueError(f"No parquet files found matching pattern '{pattern}'")

    schema = {
        "id": pl.Utf8,
        "trip_id": pl.Utf8,
        "start_date": pl.Utf8,
        "schedule_relationship": pl.Int64,
        "route_id": pl.Utf8,
        "direction_id": pl.Int64,
        "vehicle_id": pl.Utf8,
        "vehicle_label": pl.Utf8,
        "latitude": pl.Float64,
        "longitude": pl.Float64,
        "current_stop_sequence": pl.Int64,
        "current_status": pl.Int64,
        "timestamp": pl.Int64,
        "stop_id": pl.Utf8,
        "current_datetime": pl.Datetime("ns"),
    }

    df = pl.scan_parquet(matching_files, schema=schema)

    df = (
        df.with_columns(
            pl.col("current_datetime")
            .dt.convert_time_zone("America/Vancouver")
            .alias("current_datetime")
        )
        .with_columns(current_date=pl.col("current_datetime").dt.date())
        # .select(
        #     [
        #         "trip_id",
        #         "current_stop_sequence",
        #         "latitude",
        #         "longitude",
        #         "current_datetime",
        #         # "current_date",
        #     ]
        # )
        .rename({"latitude": "vehicle_latitude", "longitude": "vehicle_longitude"})
    )

    return df


# read_translink_position_polars().head().collect()

In [7]:
realtime_buckets = read_file_names(

    data_path = "../data/lambda_sync/raw_data",
    pattern = "realtime"
)

position_buckets = read_file_names(
    data_path = "../data/lambda_sync/raw_data",
    pattern = "position"
)


Searching with pattern: ../data/lambda_sync/raw_data/*_realtime.parquet
31.7075
Searching with pattern: ../data/lambda_sync/raw_data/*_position.parquet
31.37


In [43]:
for idx, bucket in enumerate(realtime_buckets):
    processed_df = process_translink_realtime_polars(bucket)
    collected_df = processed_df.collect()
    # collected_df.write_parquet(f"../data/processed/realtime/", partition_by = 'current_date')
    collected_df.write_parquet(f"../data/processed/realtime/translink_realtime_bucket_{idx}",)
    del collected_df
    # break 

In [8]:
# for idx, bucket in enumerate(position_buckets):
#     processed_df = process_translink_position_polars(bucket)
#     collected_df = processed_df.collect()
#     collected_df.write_parquet(f"../data/processed/position/translink_position_bucket_{idx}",)
#     del collected_df
#     # break 